# Scraping Data

## Scraping Civic Coding Data

### Find all pagination links

In [13]:
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup

In [30]:
base_website_to_scrape = "https://www.civic-coding.de/community-information/projekte?tx_solr%5BresultsPerPage%5D=50"

response = requests.get(base_website_to_scrape)
soup = BeautifulSoup(response.text, 'html.parser')

pagination_links = soup.find_all('a', class_='solr-ajaxified')

page_numbers = []
for link in pagination_links:
    aria_label = link.get('aria-label', '')
    if 'Gehe zu Seite:' in aria_label:
        page_num = int(aria_label.split(':')[-1].strip())
        page_numbers.append(page_num)

total_pages = max(page_numbers) if page_numbers else 1
print(f"Total pages: {total_pages}")

Total pages: 5


### Scrape all pages

In [27]:
soup_list = []

for page in range(1, total_pages + 1):
    url_to_scrape = base_website_to_scrape + "=" + str(page)
    print(f"Scraping page {page} of {total_pages}: {url_to_scrape}")
    
    response = requests.get(url_to_scrape)
    soup = BeautifulSoup(response.text, 'html.parser')
    soup_list.append(soup)

Scraping page 1 of 10: https://www.civic-coding.de/community-information/projekte?tx_solr%5Bpage%5D=1
Scraping page 2 of 10: https://www.civic-coding.de/community-information/projekte?tx_solr%5Bpage%5D=2
Scraping page 3 of 10: https://www.civic-coding.de/community-information/projekte?tx_solr%5Bpage%5D=3
Scraping page 4 of 10: https://www.civic-coding.de/community-information/projekte?tx_solr%5Bpage%5D=4
Scraping page 5 of 10: https://www.civic-coding.de/community-information/projekte?tx_solr%5Bpage%5D=5
Scraping page 6 of 10: https://www.civic-coding.de/community-information/projekte?tx_solr%5Bpage%5D=6
Scraping page 7 of 10: https://www.civic-coding.de/community-information/projekte?tx_solr%5Bpage%5D=7
Scraping page 8 of 10: https://www.civic-coding.de/community-information/projekte?tx_solr%5Bpage%5D=8
Scraping page 9 of 10: https://www.civic-coding.de/community-information/projekte?tx_solr%5Bpage%5D=9
Scraping page 10 of 10: https://www.civic-coding.de/community-information/projekte

### Get project links

In [28]:
all_link_list = []
for soup in soup_list:
    link_list = []
    project_list = soup.find_all('a', class_="position-absolute top-0 bottom-0 start-0 end-0 z-1 projectidea-link")
    
    for project in project_list:
        link = project.get('href')
        if link:  
            full_link = "https://www.civic-coding.de" + link  
            link_list.append(full_link)

    print(len(link_list))
    all_link_list.extend(link_list)  

10
10
10
10
10
10
10
10
10
10


In [29]:
len(all_link_list)

100

### Alternative: Scrape with Playwright

--> See script `civic-coding-playwright-scraping.py`

## Scraping CityLab Data

In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re

In [42]:
expected_data_set = pd.read_csv("Webscraping/Correlaid-Projektdatenbank/2026-01-19_Correlaid-Projekte-via-API_enriched.csv", sep=";")
expected_data_set

,Quelle,Projektname,Art,Einsatzbereich,Webseite-Link,Organisation,Status,Kurzzusammenfassung,Projekt-Abkürzung,Lizenz,Lizenz-Organisation
0,https://correlaid.org/daten-nutzen/projektdate...,Qualitätsanalyse von OpenStreetMap Daten für T...,"Datenanalyse, Explorative Analyse, Reporting, ...","Deutschland, Umweltschutz, Nachhaltigkeit, Öko...",https://correlaid.github.io/trinkbrunnen-analy...,"a tip: tap e.V., CorrelAid e.V.",In Betrieb,Wir haben Open Street Maps-Daten zu Trinkbrunn...,NaN,CC-BY 4.0,https://correlaid.org/
1,https://correlaid.org/daten-nutzen/projektdate...,Mithilfe von KI Wirkungsmessung skalieren,"KI Anwendung, Datenanalyse, Datenanwendung für...","Arbeit mit Kindern, Chancengleichheit, Inklusi...",NaN,"In safe hands e.V., CorrelAid e.V.",In Betrieb,"Wir haben ein Tool entwickelt, das die Auswert...",NaN,CC-BY 4.0,https://correlaid.org/
2,https://correlaid.org/daten-nutzen/projektdate...,Wertvolle Zeit sparen und Fehler vermeiden dur...,"Automatisierung von Prozessen, Datenanalyse, P...","Bildung, Wissenschaft, Forschung, Datenorienti...",NaN,"Science on Stage e.V., CorrelAid e.V.",In Betrieb,"Wir haben Skripte entwickelt, um die Datenvera...",NaN,CC-BY 4.0,https://correlaid.org/
3,https://correlaid.org/daten-nutzen/projektdate...,Automatisiertes Monitoring von Zielgruppenentw...,"Automatisierte Datenübermittlung, Reporting, D...","Kinder- und Jugendbildung, Bildungschancen, So...",NaN,"Chancenwerk e.V., CorrelAid e.V.",In Betrieb,Unterstützung von Chancenwerk mit automatisier...,NaN,CC-BY 4.0,https://correlaid.org/
4,https://correlaid.org/daten-nutzen/projektdate...,Automatisierte Fragebogenauswertung mit Genera...,Automatisierte Fragebogenauswertung mit Genera...,"Familienfreundlichkeit, Kinderbetreuung, Trans...",https://github.com/CorrelAid/workshop-babylots...,"Babylotse, CorrelAid e.V.",In Betrieb,Mithilfe von Large Language Models wertete Cor...,NaN,CC-BY 4.0,https://correlaid.org/
5,https://correlaid.org/daten-nutzen/projektdate...,Automatisiertes Qualitätsmanagement für ein Me...,"Automatisierte Datenübermittlung, Reporting, V...","Jugendbeteiligung, Jugendhilfe, Mentoring, Bil...",NaN,"Sindbad, CorrelAid e.V.",In Planung,Ein flexibles Dashboard: Mentoring-Feedback ei...,NaN,CC-BY 4.0,https://correlaid.org/
6,https://correlaid.org/daten-nutzen/projektdate...,Mit Daten zu transparenterer Demokratie,"Datenanalyse, Datenerhebung, Datenanwendung fü...","Demokratie, Transparenz, Demokratie-Wegweiser,...",https://civic-data.de/ein-wegweiser-fuer-die-d...,"Demokratie-Wegweiser, Civic Data Lab, CorrelAi...",In Betrieb,Entwicklung einer datenbasierten Lösung für da...,NaN,CC-BY 4.0,https://correlaid.org/
7,https://correlaid.org/daten-nutzen/projektdate...,Output Monitoring durch Entwicklung und Implem...,"Output Monitoring, Umfrage, Datenerhebung","Jugendarbeit, Jugendbeteiligung, Jugendhilfe, ...",https://civic-data.de/output-monitoring-ten-sing/,"CVJM-Gesamtverband in Deutschland e. V., Civic...",In Betrieb,"Bewältigung der Datenherausforderungen, denen ...",NaN,CC-BY 4.0,https://correlaid.org/
8,https://correlaid.org/daten-nutzen/projektdate...,Open Data aus Bürger- und Jugendbeteiligungspr...,"Datenerhebung, Datenmanagement, KI, Open Data,...","Demokratie, Jugendbeteiligung, Politische Bild...",https://civic-data.de/kommuki-open-data/,"Politik zum Anfassen e.V., Civic Data Lab, Cor...",In Betrieb,Wir unterstützten die Umsetzung eines Datenpro...,NaN,CC-BY 4.0,https://correlaid.org/
9,https://correlaid.org/daten-nutzen/projektdate...,Unterstützung des Relaunchs des Leerstandsmelders,"Datenanalyse, Datenerhebung, Digitale Plattfor...","Deutschland, Wohnen, Wohnungslosenhilfe, Sozia...","https://civic-data.de/leerstandsmelder/, https...","Leerstandsmelder; Gängeviertel e.V., Civic Dat...",In Betrieb,"Back- und Frontend, Contententwicklung, UX-Tes...",NaN,CC-BY 4.0,https://correlaid.org/


In [3]:
url = 'https://citylab-berlin.org/de/projects/'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

In [4]:
def assign_project_status(chunk):
    if chunk == 0:
        return "In Planung"
    else:
        return "In Betrieb"

def collect_project_categories(project_data):
    category_texts = []
    categories = project_data.find('div', class_="wpgb-block-2").find_all("a")
    for category in categories:
        category_text = category.text
        category_texts.append(category_text)

    list_as_string = ', '.join(category_texts)
    
    return list_as_string

In [5]:
project_chunks = soup.find_all('div', class_='wpgb-masonry')
projects_df=pd.DataFrame()

for chunk in range(len(project_chunks)):
    for project in project_chunks[chunk].find_all('article'): 
        quelle = project.find('a', class_="wpgb-card-layer-link").get('href')
        title = project.find("h3", class_="wpgb-block-3 wpgb-idle-scheme-1").text
        project_summary = project.find(class_="wpgb-block-6 wpgb-idle-scheme-2").text

        project_categories = collect_project_categories(project)

        # Collect data in a DataFrame
        data = {
            "Quelle": quelle,
            "Projektname": title,
            "Einsatzbereich": f"{project_categories}",
            "Webseite-Link": quelle,
            "Organisation": "CityLAB Berlin",
            "Status": assign_project_status(chunk),
            "Kurzzusammenfassung": project_summary,
            "Lizenz": "?", # TODO: Discuss with CDL Team
            "Lizenz-Organisation": "https://citylab-berlin.org"
        }
        df = pd.DataFrame(data, index=[0])
        projects_df = pd.concat([projects_df, df], ignore_index=True)

In [10]:
projects_df["Index"] = projects_df.index
projects_df

,index,Quelle,Projektname,Einsatzbereich,Webseite-Link,Organisation,Status,Kurzzusammenfassung,Lizenz,Lizenz-Organisation,Index
0,0,https://citylab-berlin.org/de/projects/baergpt/,BärGPT,"DE, Digitale Zusammenarbeit, Neue Technologien",https://citylab-berlin.org/de/projects/baergpt/,CityLAB Berlin,In Planung,Der KI-Assistent für die Berliner Landesverwal...,?,https://citylab-berlin.org,0
1,1,https://citylab-berlin.org/de/projects/fairgnu...,Fairgnügen,"DE, Digitale Zusammenarbeit, Innovative Verwal...",https://citylab-berlin.org/de/projects/fairgnu...,CityLAB Berlin,In Planung,Unsere neue Webseite hilft bei der Suche nach ...,?,https://citylab-berlin.org,1
2,2,https://citylab-berlin.org/de/projects/parla/,Parla,"DE, Digitale Zusammenarbeit, Innovative Verwal...",https://citylab-berlin.org/de/projects/parla/,CityLAB Berlin,In Planung,Das neue KI-Tool Parla macht Schriftliche Anfr...,?,https://citylab-berlin.org,2
3,3,https://citylab-berlin.org/de/projects/kiezlabor/,Kiezlabor,"DE, Digitale Zusammenarbeit, Innovative Verwal...",https://citylab-berlin.org/de/projects/kiezlabor/,CityLAB Berlin,In Planung,Das Kiezlabor bringt den prototypischen Ansatz...,?,https://citylab-berlin.org,3
4,4,https://citylab-berlin.org/de/projects/data-hu...,Data Hub Berlin,"DE, Innovative Verwaltung, Offene Daten, Open ...",https://citylab-berlin.org/de/projects/data-hu...,CityLAB Berlin,In Planung,Die zentrale Plattform für ein bessere Wertsch...,?,https://citylab-berlin.org,4
5,5,https://citylab-berlin.org/de/projects/partizi...,Partizipation Digital,"DE, Digitale Zusammenarbeit, Innovative Verwal...",https://citylab-berlin.org/de/projects/partizi...,CityLAB Berlin,In Planung,Eine prototypische Lösung soll den Ankunftspro...,?,https://citylab-berlin.org,5
6,6,https://citylab-berlin.org/de/projects/govtech...,GovTech TestLAB,"DE, Digitale Zusammenarbeit, Innovative Verwal...",https://citylab-berlin.org/de/projects/govtech...,CityLAB Berlin,In Planung,"Wir schaffen einen Raum, in dem Lösungen auspr...",?,https://citylab-berlin.org,6
7,7,https://citylab-berlin.org/de/projects/stadtla...,Stadtlabor2Go,"DE, Digitale Zusammenarbeit, Innovative Verwal...",https://citylab-berlin.org/de/projects/stadtla...,CityLAB Berlin,In Planung,"Stadtlabor2Go zeigt, wie Städte Digitalisierun...",?,https://citylab-berlin.org,7
8,8,https://citylab-berlin.org/de/projects/giess-d...,Gieß den Kiez,"DE, Energie und Nachhaltigkeit, Offene Daten, ...",https://citylab-berlin.org/de/projects/giess-d...,CityLAB Berlin,In Planung,Gieß den Kiez ist eine Plattform zur Koordinie...,?,https://citylab-berlin.org,8
9,9,https://citylab-berlin.org/de/projects/klimada...,KlimaDashboard Xhain,"DE, Digitale Zusammenarbeit, Energie und Nachh...",https://citylab-berlin.org/de/projects/klimada...,CityLAB Berlin,In Planung,Mit dem Klimadashboard werden Klimadaten trans...,?,https://citylab-berlin.org,9


In [ ]:

laufend_chunk = first_chunks[0].find_all('article')
abgeschlossen_chunk = first_chunks[1].find_all('article')

In [ ]:
laufend_dict = {}

for tile in laufend_chunk: 
    sub_domain_link = tile.find('a', class_="wpgb-card-layer-link").get('href')
    h3 = tile.find('h3', class_="wpgb-block-3 wpgb-idle-scheme-1")
    if h3:
        textlink = h3.find('a')
        if textlink:
            title = textlink.text
    
    category_texts = []
    cat = tile.find('div', class_="wpgb-block-2")
    if cat: 
        categories = cat.find_all('a')
        if categories:
            for category in categories:
                category_text = category.text
                category_texts.append(category_text)
    
    project_summary = tile.find(class_="wpgb-block-6 wpgb-idle-scheme-2").text
    
    laufend_dict[title] = {
        "Quelle" : sub_domain_link,
        "Einsatzbereich" : category_texts,
        "Kurzzusammenfassung" : project_summary
    } 

for key in list(laufend_dict.keys()):
    split_string = re.split(r'[\\]', key)
    if split_string:  # Check if the split string has elements
        new_key = split_string[0].strip()
    laufend_dict[new_key] = laufend_dict.pop(key)

laufend_dict

{'BärGPT': {'sub-link': 'https://citylab-berlin.org/de/projects/baergpt/',
  'Einsatzbereich': ['Digitale Zusammenarbeit', 'Neue Technologien'],
  'Kurzzusammenfassung': 'Der KI-Assistent für die Berliner Landesverwaltung.'},
 'Fairgnügen': {'sub-link': 'https://citylab-berlin.org/de/projects/fairgnuegen/',
  'Einsatzbereich': ['Digitale Zusammenarbeit',
   'Innovative Verwaltung',
   'Prototyping'],
  'Kurzzusammenfassung': 'Unsere neue Webseite hilft bei der Suche nach über 400 kostenlosen und ermäßigten Angeboten.'},
 'Parla': {'sub-link': 'https://citylab-berlin.org/de/projects/parla/',
  'Einsatzbereich': ['Digitale Zusammenarbeit',
   'Innovative Verwaltung',
   'Neue Technologien'],
  'Kurzzusammenfassung': 'Das neue KI-Tool Parla macht Schriftliche Anfragen durchsuchbar. '},
 'Kiezlabor': {'sub-link': 'https://citylab-berlin.org/de/projects/kiezlabor/',
  'Einsatzbereich': ['Digitale Zusammenarbeit',
   'Innovative Verwaltung',
   'Offene Daten, offene Städte',
   'Smart Cities

In [ ]:
abgeschlossen_dict = {}

for tile in abgeschlossen_chunk: 
    sub_domain_link = tile.find('a', class_="wpgb-card-layer-link").get('href')
    h3 = tile.find('h3', class_="wpgb-block-3 wpgb-idle-scheme-1")
    if h3:
        textlink = h3.find('a')
        if textlink:
            title = textlink.text

    category_texts = []
    cat = tile.find('div', class_="wpgb-block-2")
    if cat: 
        categories = cat.find_all('a')
        if categories:
            for category in categories:
                category_text = category.text
                category_texts.append(category_text)

    abgeschlossen_dict[title] = {
        "sub-link" : sub_domain_link,
        "Einsatzbereich" : category_texts
    } 

for key in list(abgeschlossen_dict.keys()):
    split_string = re.split(r'[\–\-\:]', key)
    if split_string:  # Check if the split string has elements
        new_key = split_string[0].strip()
    abgeschlossen_dict[new_key] = abgeschlossen_dict.pop(key)

abgeschlossen_dict

{'Digital Vereint': {'sub-link': 'https://citylab-berlin.org/de/projects/digitalvereint/',
  'Einsatzbereich': ['Smart Cities für alle']},
 'Quantified Trees (Qtrees)': {'sub-link': 'https://citylab-berlin.org/de/projects/qtrees/',
  'Einsatzbereich': ['Energie und Nachhaltigkeit']},
 'Verwaltungsdashboard': {'sub-link': 'https://citylab-berlin.org/de/projects/afs-dashboard/',
  'Einsatzbereich': ['Innovative Verwaltung']},
 'Stadtpuls': {'sub-link': 'https://citylab-berlin.org/de/projects/stadtpuls/',
  'Einsatzbereich': ['Offene Daten, offene Städte', 'Smart Cities für alle']},
 'Pixelwelt': {'sub-link': 'https://citylab-berlin.org/de/projects/pixelwelt/',
  'Einsatzbereich': ['Digitale Zusammenarbeit']},
 'Prototypenwerkstatt': {'sub-link': 'https://citylab-berlin.org/de/projects/prototypenwerkstatt/',
  'Einsatzbereich': ['Innovative Verwaltung']},
 'Prozessanalyse Radinfrastruktur (PARI)': {'sub-link': 'https://citylab-berlin.org/de/projects/prozessanalyse-radinfrastruktur-pari/',

# Code for further pipeline

## Scrape regular data

In [5]:
%load_ext autoreload
%autoreload 2

from Webscraping.Citylab_Berlin.scrape_projects_from_citylab_website import scrape_citylab_berlin
from Webscraping.get_ai_data import enrich_projects_data_with_ai

# Scrape CityLAB Berlin Projects
citylab_berlin_projects = scrape_citylab_berlin(
		url="https://citylab-berlin.org/de/projects/",
		save_to_csv=False
	)

citylab_berlin_projects_enriched = enrich_projects_data_with_ai(citylab_berlin_projects)
citylab_berlin_projects_enriched

# TODO: CodeFor Scraping
# TODO: Civic Coding Scraping

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Successfully retrieved 40 projects from the CityLAB Berlin website.
Processing 40 rows...
[1/40] https://citylab-berlin.org/de/projects/baergpt/
  -> Attempt 1: Using model meta-llama/llama-4-scout-17b-16e-instruct
AI result: {'Projekt-Abkürzung': 'BärGPT', 'Art': 'KI-Anwendung, Chatbot, Dokumentenverwaltung', 'Einsatzbereich': 'Verwaltung, Öffentliche Verwaltung, Berliner Landesverwaltung'}
[2/40] https://citylab-berlin.org/de/projects/fairgnuegen/
  -> Attempt 1: Using model meta-llama/llama-4-scout-17b-16e-instruct
AI result: {'Projekt-Abkürzung': 'Fairgnügen', 'Art': 'Analyse von Sensordaten und ML, Automatisierte Datenübermittlung, Beratung, Datenanalyse, Datenerhebung, Datenanwendung für Öffentlichkeit, Datenstandards, Datensatz und Visualisierung, Digitale Plattform, Entscheidungsassistent, Interaktive App, Interaktive Karte, Interaktiver Fragebogen, Interne Datenanwendung, KI Anwendung, Kart

,Index,Quelle,Projektname,Einsatzbereich,Webseite-Link,Organisation,Status,Kurzzusammenfassung,Lizenz,Lizenz-Organisation,Projekt-Abkürzung,Art
0,0,https://citylab-berlin.org/de/projects/baergpt/,BärGPT,"Verwaltung, Öffentliche Verwaltung, Berliner L...",https://citylab-berlin.org/de/projects/baergpt/,CityLAB Berlin,In Planung,Der KI-Assistent für die Berliner Landesverwal...,CC BY-NC-SA,https://citylab-berlin.org,BärGPT,"KI-Anwendung, Chatbot, Dokumentenverwaltung"
1,1,https://citylab-berlin.org/de/projects/fairgnu...,Fairgnügen,"Armut, Barrierefreiheit, Beratung, Chancenglei...",https://citylab-berlin.org/de/projects/fairgnu...,CityLAB Berlin,In Planung,Unsere neue Webseite hilft bei der Suche nach ...,CC BY-NC-SA,https://citylab-berlin.org,Fairgnügen,"Analyse von Sensordaten und ML, Automatisierte..."
2,2,https://citylab-berlin.org/de/projects/parla/,Parla,"Verwaltung, Politik, Informationsmanagement",https://citylab-berlin.org/de/projects/parla/,CityLAB Berlin,In Planung,Das neue KI-Tool Parla macht Schriftliche Anfr...,CC BY-NC-SA,https://citylab-berlin.org,Parla,"KI Anwendung, Wissensmanagement, Verwaltungsdo..."
3,3,https://citylab-berlin.org/de/projects/kiezlabor/,Kiezlabor,"Berlin, Deutschland, Urban Development, Commun...",https://citylab-berlin.org/de/projects/kiezlabor/,CityLAB Berlin,In Planung,Das Kiezlabor bringt den prototypischen Ansatz...,CC BY-NC-SA,https://citylab-berlin.org,Kiezlabor,"Partizipation, Stadtplanung, Smart City, Digit..."
4,4,https://citylab-berlin.org/de/projects/data-hu...,Data Hub Berlin,"Verwaltung, Stadtentwicklung, Forschung",https://citylab-berlin.org/de/projects/data-hu...,CityLAB Berlin,In Planung,Die zentrale Plattform für ein bessere Wertsch...,CC BY-NC-SA,https://citylab-berlin.org,Data Hub Berlin,"Datenplattform, Open-Source, Datenmanagement, ..."
5,5,https://citylab-berlin.org/de/projects/partizi...,Partizipation Digital,"Integration, Partizipation, Migration, Inklusi...",https://citylab-berlin.org/de/projects/partizi...,CityLAB Berlin,In Planung,Eine prototypische Lösung soll den Ankunftspro...,CC BY-NC-SA,https://citylab-berlin.org,PaDi,"Digitale Plattform, Interaktive App, Interakti..."
6,6,https://citylab-berlin.org/de/projects/govtech...,GovTech TestLAB,"Berliner Verwaltung, Öffentliche Verwaltung, D...",https://citylab-berlin.org/de/projects/govtech...,CityLAB Berlin,In Planung,"Wir schaffen einen Raum, in dem Lösungen auspr...",CC BY-NC-SA,https://citylab-berlin.org,GovTech TestLAB,"Testlabor, GovTech-Anwendungen, Verwaltungsdig..."
7,7,https://citylab-berlin.org/de/projects/stadtla...,Stadtlabor2Go,"Stadtplanung, Digitalisierung, Bürgerbeteiligu...",https://citylab-berlin.org/de/projects/stadtla...,CityLAB Berlin,In Planung,"Stadtlabor2Go zeigt, wie Städte Digitalisierun...",CC BY-NC-SA,https://citylab-berlin.org,Stadtlabor2Go,"Analyse von Sensordaten und ML, Automatisierte..."
8,8,https://citylab-berlin.org/de/projects/giess-d...,Gieß den Kiez,"Umweltschutz, Stadtplanung, Bürgerengagement, ...",https://citylab-berlin.org/de/projects/giess-d...,CityLAB Berlin,In Planung,Gieß den Kiez ist eine Plattform zur Koordinie...,CC BY-NC-SA,https://citylab-berlin.org,Gieß den Kiez,"Interaktive Plattform, Open Data, Datenvisuali..."
9,9,https://citylab-berlin.org/de/projects/klimada...,KlimaDashboard Xhain,"Klimaschutz, Stadtplanung, Bürgerbeteiligung, ...",https://citylab-berlin.org/de/projects/klimada...,CityLAB Berlin,In Planung,Mit dem Klimadashboard werden Klimadaten trans...,CC BY-NC-SA,https://citylab-berlin.org,KlimaDashboard Xhain,"Interaktive Visualisierung, Datenanalyse, Klim..."


## Source Manual Data